# Sync Author Curations

Syncs author curations (claim/remove works, set display_name/full_name) from the users Heroku Postgres database to local Databricks Delta tables.

**Sources** (typed views over `openalex_users.public.curations`):
- `work_author_curation_claims` — claim a work for an author at a specific named slot (`property = 'authorships[raw_author_name="<name>"].author.id'`); exposes `raw_author_name TEXT` extracted from the property string
- `work_author_curation_removals` — non-positional sticky disclaim (`property = 'authorships.author.id'`)
- `author_curations_set_display_name` — set a display_name for an author profile
- `author_curations_set_full_name` — set a full_name (used in matching) for an author profile

**Targets** (split by action / consumer):
- `openalex.works.work_author_curation_claims` — flat per-curation rows; `raw_author_name` arrives from the source view (name is the anchor for the apply MERGE in `ApplyWorkAuthorCurations`)
- `openalex.works.work_author_curation_removals` — flat per-curation rows; non-positional
- `openalex.authors.author_names_curations` — per-author display_name / full_name (applied during profile assembly / matching)

Modeled on `SyncRasCurations.ipynb` with a guardrail cell added before each MERGE so a silent empty/short source can't mass-delete the Delta table. The `WHEN NOT MATCHED BY SOURCE THEN DELETE` clause is what propagates user-initiated retractions (`DELETE /curations/{id}`), so it stays — the guardrail just refuses to fire it on a suspicious source.

**Dep ordering**: this notebook no longer needs `work_authors` data at sync time (`raw_author_name` arrives on the source row, no name-capture join), so `Sync_Author_Curations` runs after `Works_Base` rather than after `Author_Affiliations` (see `walden_end2end.yaml`).

**Apply semantics (new design, 2026-05-11)**: claim and removal Delta tables have **no `applied_at` column** — the apply notebook runs every cycle as a post-`MatchAuthors` override (idempotent MERGEs on `(work_id, raw_author_name)` and `(work_id, author_id)` respectively). There is no block-list in `MatchAuthors` — disclaim is enforced solely by the apply step re-NULLing the slot every cycle.

**v0 application status**: works curations have apply logic in `ApplyWorkAuthorCurations`. Names curations sync to Delta but are not yet applied (`set` action on `entity='author'` is currently rejected by `POST /curations` validation, so the names views and target will stay empty until producer-side support lands).

## Create target tables (idempotent)

In [ ]:
%sql
-- Claim-side curations: name-anchored. raw_author_name arrives on the source
-- row (extracted by the work_author_curation_claims view from the property
-- string) and is the join key the apply MERGE uses to find the right
-- work_authors slot. No applied_at — apply runs every cycle as a
-- post-MatchAuthors override (idempotent MERGE on (work_id, raw_author_name)).
CREATE TABLE IF NOT EXISTS openalex.works.work_author_curation_claims (
    curation_id        STRING NOT NULL,
    user_id            STRING NOT NULL,
    author_id          BIGINT NOT NULL,
    work_id            BIGINT NOT NULL,
    raw_author_name    STRING NOT NULL,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP
)
USING DELTA
CLUSTER BY (work_id)

In [ ]:
%sql
-- Removal-side curations: non-positional sticky disclaim. The apply step NULLs
-- author_id at every (work_id, author_id) slot every cycle as a
-- post-MatchAuthors override — there is no block-list in MatchAuthors.
-- No applied_at — apply is idempotent.
CREATE TABLE IF NOT EXISTS openalex.works.work_author_curation_removals (
    curation_id        STRING NOT NULL,
    user_id            STRING NOT NULL,
    author_id          BIGINT NOT NULL,
    work_id            BIGINT NOT NULL,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

In [ ]:
%sql
-- Per-author display_name / full_name curations.
-- Latest-wins by source `created` when multiple curators set the same property
-- (aggregation handled in the MERGE source query below).
CREATE TABLE IF NOT EXISTS openalex.authors.author_names_curations (
    author_id             BIGINT NOT NULL,
    curated_display_name  STRING,
    curated_full_name     STRING,
    updated_datetime      TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

## Sync works curations

In [ ]:
%sql
-- Guardrail: refuse to MERGE if the source has unexpectedly few rows.
-- Counts cover BOTH the claims and removals views (we want to fail-stop on
-- either side emptying out, since a single guardrail cell precedes both MERGEs).
-- The empty-source check is conditional on the target already holding rows,
-- so the legitimate startup state (both 0) doesn't fail; only "we had data
-- and now the source is empty" fails.
-- Set guardrails_override=true on the job to bypass the decline check
-- (the empty-when-target-nonempty check is unconditional).
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (
  SELECT COUNT(*) FROM (
    SELECT curation_id FROM openalex_users.public.work_author_curation_claims
    UNION ALL
    SELECT curation_id FROM openalex_users.public.work_author_curation_removals
  )
);
SET VARIABLE current_count = (
  SELECT (SELECT COUNT(*) FROM openalex.works.work_author_curation_claims)
       + (SELECT COUNT(*) FROM openalex.works.work_author_curation_removals)
);

SELECT
  new_count AS new_curations,
  current_count AS current_curations,
  ROUND(new_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 work-author curations but targets hold ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source views before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source curation count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job to bypass.'))
END;

In [ ]:
%sql
-- Preview what will be synced: each curation row from the source views.
-- raw_author_name arrives on the source row for claims (extracted from the
-- property string by the Postgres view); removals are non-positional.
SELECT
    'claim'  AS action_kind,
    curation_id, user_id, author_id, work_id,
    raw_author_name,
    created
FROM openalex_users.public.work_author_curation_claims
UNION ALL
SELECT
    'remove' AS action_kind,
    curation_id, user_id, author_id, work_id,
    CAST(NULL AS STRING) AS raw_author_name,
    created
FROM openalex_users.public.work_author_curation_removals

In [ ]:
%sql
-- MERGE claim-side curations. raw_author_name arrives on the source row from
-- the Postgres view (extracted from the property string). No work_authors
-- join (no name-capture needed), no applied_at preservation — apply runs
-- every cycle and is idempotent on (work_id, raw_author_name).
MERGE INTO openalex.works.work_author_curation_claims AS target
USING (
    SELECT
        curation_id, user_id, author_id, work_id, raw_author_name, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.work_author_curation_claims
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [ ]:
%sql
-- MERGE removal-side curations. Non-positional, no applied_at preservation —
-- apply runs every cycle as a post-MatchAuthors override and is idempotent
-- on (work_id, author_id).
MERGE INTO openalex.works.work_author_curation_removals AS target
USING (
    SELECT
        curation_id, user_id, author_id, work_id, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.work_author_curation_removals
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Sync names curations

In [ ]:
%sql
-- Guardrail: same shape as the works guardrail. The source for names is the
-- UNION of distinct author_ids across the two `set_*name` views.
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (
  SELECT COUNT(DISTINCT author_id) FROM (
    SELECT author_id FROM openalex_users.public.author_curations_set_display_name
    UNION
    SELECT author_id FROM openalex_users.public.author_curations_set_full_name
  )
);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.authors.author_names_curations);

SELECT
  new_count AS new_authors,
  current_count AS current_authors,
  ROUND(new_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 author name-curations but target has ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source views before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source author count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job to bypass.'))
END;

In [ ]:
%sql
-- Preview what will be synced (latest-wins per author for each name property)
WITH latest_display_name AS (
  SELECT author_id, new_display_name AS curated_display_name
  FROM (
    SELECT
      author_id,
      new_display_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_curations_set_display_name
  )
  WHERE rn = 1
),
latest_full_name AS (
  SELECT author_id, new_full_name AS curated_full_name
  FROM (
    SELECT
      author_id,
      new_full_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_curations_set_full_name
  )
  WHERE rn = 1
)
SELECT
  COALESCE(d.author_id, f.author_id) AS author_id,
  d.curated_display_name,
  f.curated_full_name
FROM latest_display_name d
FULL OUTER JOIN latest_full_name f USING (author_id)

In [ ]:
%sql
-- MERGE names curations (handles inserts, updates, AND deletes via NOT MATCHED BY SOURCE)
MERGE INTO openalex.authors.author_names_curations AS target
USING (
  WITH latest_display_name AS (
    SELECT author_id, new_display_name AS curated_display_name
    FROM (
      SELECT
        author_id,
        new_display_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_curations_set_display_name
    )
    WHERE rn = 1
  ),
  latest_full_name AS (
    SELECT author_id, new_full_name AS curated_full_name
    FROM (
      SELECT
        author_id,
        new_full_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_curations_set_full_name
    )
    WHERE rn = 1
  )
  SELECT
    COALESCE(d.author_id, f.author_id) AS author_id,
    d.curated_display_name,
    f.curated_full_name,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM latest_display_name d
  FULL OUTER JOIN latest_full_name f USING (author_id)
) AS source
ON target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
-- Works targets: row counts and sync timestamps
SELECT
  (SELECT COUNT(*)              FROM openalex.works.work_author_curation_claims)   AS claim_curation_rows,
  (SELECT COUNT(*)              FROM openalex.works.work_author_curation_removals) AS removal_curation_rows,
  (SELECT MAX(updated_datetime) FROM openalex.works.work_author_curation_claims)   AS claim_last_sync,
  (SELECT MAX(updated_datetime) FROM openalex.works.work_author_curation_removals) AS removal_last_sync

In [ ]:
%sql
-- Sample of recent curations from both sides (action_kind discriminator)
SELECT 'claim' AS action_kind,
       curation_id, user_id, author_id, work_id, raw_author_name, created, updated_datetime
FROM openalex.works.work_author_curation_claims
UNION ALL
SELECT 'remove' AS action_kind,
       curation_id, user_id, author_id, work_id,
       CAST(NULL AS STRING) AS raw_author_name,
       created, updated_datetime
FROM openalex.works.work_author_curation_removals
ORDER BY updated_datetime DESC
LIMIT 100

In [ ]:
%sql
-- Names target: row counts
SELECT
  COUNT(*)                                                AS total_curated_authors,
  COUNT(curated_display_name)                             AS total_display_names,
  COUNT(curated_full_name)                                AS total_full_names,
  MAX(updated_datetime)                                   AS last_sync
FROM openalex.authors.author_names_curations

In [ ]:
%sql
-- Sample of curated authors (names)
SELECT * FROM openalex.authors.author_names_curations
ORDER BY updated_datetime DESC
LIMIT 100